# scicp — Fine-tune MiniLM on Scripture

Fine-tunes `all-MiniLM-L6-v2` on 62k topical-guide (topic → verse) pairs.
With a T4 GPU this takes **~15–20 minutes**.

**Steps:**
1. Run all cells top to bottom
2. Upload `training-pairs.json` when prompted (or mount Google Drive)
3. The final cell downloads `scripture-minilm.zip` — unzip into `resources/models/`
4. Run `python3 scripts/rebake-embeddings.py` on your machine to re-encode all verses

In [ ]:
# ── 1. Install dependencies ──────────────────────────────────────────────────
!pip install -q sentence-transformers datasets accelerate

In [ ]:
# ── 2. Check GPU ─────────────────────────────────────────────────────────────
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

CUDA: True
GPU: Tesla T4
VRAM: 15.6 GB


In [ ]:
# ── 3. Upload training-pairs.json ────────────────────────────────────────────
# Option A: upload from your machine
# from google.colab import files
# uploaded = files.upload()   # select resources/training-pairs.json

# Option B: if you prefer Google Drive, comment out the two lines above and run:
from google.colab import drive
drive.mount('/content/drive')
PAIRS_PATH = '/content/drive/MyDrive/scicp/training-pairs.json'




Mounted at /content/drive


In [ ]:
# ── 4. Load pairs ────────────────────────────────────────────────────────────
# ── 4. Load pairs ────────────────────────────────────────────────────────────
import json, random

# PAIRS_PATH = 'training-pairs.json'  # ← uploaded filename

with open(PAIRS_PATH) as f:
    pairs = json.load(f)

random.seed(42)
random.shuffle(pairs)

print(f'Loaded {len(pairs):,} pairs')
print('Sample:', pairs[0])

Loaded 252,485 pairs
Sample: {'anchor': 'Sword', 'positive': 'And there shall be earthquakes also in divers places, and many desolations; yet men will harden their hearts against me, and they will take up the sword, one against another, and they will kill one another.'}


In [ ]:
# ── 5. Fine-tune ─────────────────────────────────────────────────────────────
from sentence_transformers import SentenceTransformer, losses
from sentence_transformers.training_args import SentenceTransformerTrainingArguments
from sentence_transformers.trainer import SentenceTransformerTrainer
from datasets import Dataset
import time

BASE_MODEL  = 'sentence-transformers/all-MiniLM-L6-v2'
OUT_DIR     = '/content/scripture-minilm'
BATCH_SIZE  = 128    # T4 can handle 128 comfortably for MiniLM-L6
EPOCHS      = 4
WARMUP_FRAC = 0.1

split = int(len(pairs) * 0.9)
train_pairs = pairs[:split]
val_pairs   = pairs[split:]

train_ds = Dataset.from_dict({
    'anchor':   [p['anchor']   for p in train_pairs],
    'positive': [p['positive'] for p in train_pairs],
})
val_ds = Dataset.from_dict({
    'anchor':   [p['anchor']   for p in val_pairs],
    'positive': [p['positive'] for p in val_pairs],
})

print(f'train={len(train_ds):,}  val={len(val_ds):,}')

model = SentenceTransformer(BASE_MODEL)
loss  = losses.MultipleNegativesRankingLoss(model)

steps_per_epoch = len(train_ds) // BATCH_SIZE
warmup_steps    = int(steps_per_epoch * EPOCHS * WARMUP_FRAC)

args = SentenceTransformerTrainingArguments(
    output_dir=OUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    warmup_steps=warmup_steps,
    eval_strategy='epoch',
    save_strategy='best',
    load_best_model_at_end=True,
    logging_steps=50,
    fp16=torch.cuda.is_available(),   # fp16 on GPU, fp32 on CPU
    bf16=False,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
)

trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    loss=loss,
)

t0 = time.time()
trainer.train()
print(f'\nDone in {(time.time()-t0)/60:.1f} min')

train=314,541  val=34,950


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Done in 60.9 min


In [ ]:
# ── 6. Save model to Google Drive ───────────────────────────────────────────
import os

# Saves into the same Drive folder as training-pairs.json
DRIVE_OUT = '/content/drive/MyDrive/scicp/scripture-minilm'
os.makedirs(DRIVE_OUT, exist_ok=True)
model.save(DRIVE_OUT)
print(f'Model saved to Drive: {DRIVE_OUT}')
print('Files:', os.listdir(DRIVE_OUT))

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to Drive: /content/drive/MyDrive/scicp/scripture-minilm
Files: ['config_sentence_transformers.json', 'config.json', 'model.safetensors', 'tokenizer_config.json', 'tokenizer.json', 'sentence_bert_config.json', '1_Pooling', '2_Normalize', 'modules.json', 'README.md']


## After training

The fine-tuned model is saved to **My Drive → scicp/scripture-minilm/**.

On your local machine:

```bash
# 1. Download the model folder from Google Drive to:
#    resources/models/scripture-minilm/

# 2. Re-encode all 41k verses with the fine-tuned model (~3 min)
python3 scripts/rebake-embeddings.py

# 3. Rebuild cluster labels (centroids have changed)
node scripts/prebake-cluster-labels.js

# 4. Restart the server
npm run dev
```

The server loads embeddings from `verse-embeddings.db` at startup — no code change needed.

In [ ]:
import os, shutil

DRIVE_OUT = '/content/drive/MyDrive/scicp/scripture-minilm'
os.makedirs(DRIVE_OUT, exist_ok=True)
shutil.copytree('/content/scripture-minilm', DRIVE_OUT, dirs_exist_ok=True)
print('Files saved:', os.listdir(DRIVE_OUT))

Files saved: ['config_sentence_transformers.json', 'config.json', 'model.safetensors', 'tokenizer_config.json', 'tokenizer.json', 'sentence_bert_config.json', '1_Pooling', '2_Normalize', 'modules.json', 'README.md', 'checkpoint-9832', 'checkpoint-4916', 'checkpoint-7374', 'checkpoint-2458']


In [ ]:
import shutil, os

shutil.make_archive('/content/scripture-minilm', 'zip', '/content/scripture-minilm')
shutil.copy('/content/scripture-minilm.zip', '/content/drive/MyDrive/scicp/scripture-minilm.zip')
print('Saved to Drive:', os.path.getsize('/content/drive/MyDrive/scicp/scripture-minilm.zip') // 1024 // 1024, 'MB')